**Table of contents**<a id='toc0_'></a>    
- [1. LSTM: Why it is needed and how the four gates work](#toc1_1_)    
    - [Step 1: Dicide what to forget (Forget Gate)](#toc1_1_1_)    
    - [Step 2: Decide what new information to Add (Input gate + Candidate memory)](#toc1_1_2_)    
    - [Step 3: Update the Cell State (Memory Highway)](#toc1_1_3_)    
    - [Step 4: Generate the Output (Output Gate)](#toc1_1_4_)    
    - [LSTM cell architecture](#toc1_1_5_)    
- [2. RNN basics: LSTM cell, Encoder, Decoder, Seq2Seq baseline](#toc2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[1. LSTM: Why it is needed and how the four gates work](#toc0_)

A vanilla RNN can suffer from *vanishing gradients* on long sequences: the gradient becomes smaller across time steps, making it difficult to preserve information from the beginning of a sentence.

LSTM adds a separate memory path called the **cell state** $c_t$. This state is updated through an additive path, which helps gradients remain more stable. At each time step, an LSTM uses four components:

- **Forget gate** $f_t$: decides how much information to keep from $c_{t-1}$; sigmoid output is in $(0, 1)$.
- **Input gate** $i_t$: decides how much new information to write into the cell state; uses sigmoid.
- **Candidate state** $\tilde{c}_t$: new content that may be written into memory; uses $\tanh$, whose output is in $(-1, 1)$.
- **Output gate** $o_t$: decides how much information from the cell state becomes the hidden state; uses sigmoid.

For current input $x_t$, previous hidden state $h_{t-1}$, and previous cell state $c_{t-1}$:

$$
\begin{aligned}
i_t &= \sigma(x_t W_{xi} + h_{t-1} W_{hi} + b_i), \\
f_t &= \sigma(x_t W_{xf} + h_{t-1} W_{hf} + b_f), \\
\tilde{c}_t &= \tanh(x_t W_{xc} + h_{t-1} W_{hc} + b_c), \\
o_t &= \sigma(x_t W_{xo} + h_{t-1} W_{ho} + b_o).
\end{aligned}
$$

The memory and hidden state are then updated as follows:

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

$$
h_t = o_t \odot \tanh(c_t)
$$

The addition in the $c_t$ equation creates a direct memory path. This is the key mechanism that helps LSTM reduce the vanishing-gradient problem compared with a vanilla RNN.

### <a id='toc1_1_1_'></a>[Step 1: Dicide what to forget (Forget Gate)](#toc0_)

The forget gate takes the previous hidden state $h_{t-1}$ and the current input $x_{t}$.

It produces values between 0 and 1 using a sigmoid.

These values determine how much of the previous cell state $c_{t - 1}$ should be kept.
 
Values near 1 mean the information is reetained.

Values near 0 mean the information is discarded.

This mechanism helps remove outdated or irrelevant context from long sequences.

### <a id='toc1_1_2_'></a>[Step 2: Decide what new information to Add (Input gate + Candidate memory)](#toc0_)

The input gate determines how much of the new information should be written into the memory.

The candidate memory creates a new vector of potential information using a Tanh.

The input gate filters the candidate memory before it is added to the cell state.

This ensures that only relevant new information updates the internal memory.

### <a id='toc1_1_3_'></a>[Step 3: Update the Cell State (Memory Highway)](#toc0_)

The updated cell state is formed by combining the scaled previous memory and the filtered new memory.

The forget gate reduces or removes parts of the old memory.

The input gate and candidate memory add new information to fill in or modify the cell state.

Additive updates help gradients flow more consistently, preventing the vanishing gradient problem.

This stable memory path enables learning complex long-range patterns.

### <a id='toc1_1_4_'></a>[Step 4: Generate the Output (Output Gate)](#toc0_)

The output gate determines how much of the updated cell state influences the hidden state $h_{t}$.

A sigmoid activation computes this output filter.

The filtered value is merged with the tanh-transformed cell state to produce the final hidden state.

The hidden state serves as the LSTM’s output at time step t.

The same hidden state is passed forward to the next time step for continued processing.



### <a id='toc1_1_5_'></a>[LSTM cell architecture](#toc0_)

![LSTM cell architecture](../../images/architecture.png)

+ Yellow nodes represent element-wise multiplication ($\odot$); 
+ Green node represents element-wise addition.

$$
\begin{aligned}
i_t &= \sigma(x_t W_{xi} + h_{t-1} W_{hi} + b_i), \\
f_t &= \sigma(x_t W_{xf} + h_{t-1} W_{hf} + b_f), \\
\tilde{c}_t &= \tanh(x_t W_{xc} + h_{t-1} W_{hc} + b_c), \\
o_t &= \sigma(x_t W_{xo} + h_{t-1} W_{ho} + b_o).
\end{aligned}
$$

https://codingowen.github.io/projects/lstm_from_scratch/

# <a id='toc2_'></a>[2. RNN basics: LSTM cell, Encoder, Decoder, Seq2Seq baseline](#toc0_)

xây baseline Seq2Seq LSTM **KHÔNG có Attention**


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


import pickle
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from src.vocab import Vocab

torch.manual_seed(42)   # cố định random để kết quả LẶP LẠI ĐƯỢC mỗi lần chạy
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on:", device)

Running on: cuda


In [3]:
DATA_DIR = 'D:/AIResearcher/MachineTraslation/data/processed'

with open(f'{DATA_DIR}/en_vocab.pkl', 'rb') as f:
    en_vocab = pickle.load(f)
with open(f'{DATA_DIR}/vi_vocab.pkl', 'rb') as f:
    vi_vocab = pickle.load(f)
with open(f'{DATA_DIR}/train_ids.pkl', 'rb') as f:
    train_ids = pickle.load(f)
with open(f'{DATA_DIR}/dev_ids.pkl', 'rb') as f:
    dev_ids = pickle.load(f)
with open(f'{DATA_DIR}/test_ids.pkl', 'rb') as f:
    test_ids = pickle.load(f)

In [4]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs # list[(src_ids: list[int], tgt_ids: list[int])]
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

In [5]:
train_dataset = TranslationDataset(train_ids) # list[(src_ids: list[int], tgt_ids: list[int])]
dev_dataset   = TranslationDataset(dev_ids)
test_dataset  = TranslationDataset(test_ids)

In [6]:
PAD_IDX = 0 # đã định nghĩa trong Vocab: {'<pad>': 0, ...}
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_lengths = torch.tensor([len(s) for s in src_batch])
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    # tgt_padded: [<bos>, i, am, a, student, <eos>]
    # tgt_input  = [<bos>, i, am, a, student] bỏ token CUỐI
    # tgt_output = [i, am, a, student, <eos>] bỏ token ĐẦU
    tgt_input  = tgt_padded[:, :-1]   # INPUT cho decoder
    tgt_output = tgt_padded[:, 1:]    # TARGET để tính loss
    return src_padded, src_lengths, tgt_input, tgt_output


In [7]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(type(train_loader))

<class 'torch.utils.data.dataloader.DataLoader'>
